# Benchmark Performance Analysis

This notebook analyzes the performance characteristics of causal inference methods
across different sample sizes, method families, and computational complexity.

**Session 131**: Performance Benchmarks Infrastructure

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup
sys.path.insert(0, str(Path.cwd().parent.parent))
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

## 1. Load Benchmark Data

In [ ]:
# Load baseline if available, otherwise run benchmarks
baseline_path = Path("../../benchmarks/golden/benchmark_baseline.json")

if baseline_path.exists():
    with open(baseline_path) as f:
        baseline = json.load(f)
    print(f"Loaded baseline from {baseline_path}")
    print(f"Generated: {baseline['metadata']['generated_at']}")
else:
    print("No baseline found. Run `python benchmarks/golden/generate_baseline.py` first.")
    baseline = None

In [ ]:
def baseline_to_dataframe(baseline: dict) -> pd.DataFrame:
    """Convert baseline JSON to DataFrame."""
    rows = []
    for family, methods in baseline.get("families", {}).items():
        for method, sizes in methods.items():
            for n, metrics in sizes.items():
                if "error" not in metrics:
                    rows.append({
                        "family": family,
                        "method": method,
                        "n": int(n),
                        "median_ms": metrics["median_ms"],
                        "min_ms": metrics["min_ms"],
                        "max_ms": metrics["max_ms"],
                        "std_ms": metrics["std_ms"],
                        "memory_kb": metrics["memory_kb"],
                        "speed_category": metrics["speed_category"],
                    })
    return pd.DataFrame(rows)

if baseline:
    df = baseline_to_dataframe(baseline)
    print(f"Loaded {len(df)} benchmark results")
    print(f"Families: {df['family'].nunique()}")
    print(f"Methods: {df['method'].nunique()}")
    df.head()

## 2. Performance Heatmap by Method Family

In [ ]:
def plot_performance_heatmap(df: pd.DataFrame, sample_size: int = 1000):
    """Create heatmap of method timings by family."""
    subset = df[df["n"] == sample_size].copy()
    
    # Pivot for heatmap
    pivot = subset.pivot_table(
        index="method", 
        columns="family", 
        values="median_ms",
        aggfunc="first"
    )
    
    # Log scale for better visualization
    log_pivot = np.log10(pivot.fillna(0) + 1)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(
        log_pivot,
        cmap="YlOrRd",
        annot=pivot.round(1),
        fmt=".1f",
        mask=pivot.isna(),
        ax=ax,
        cbar_kws={"label": "log10(ms)"}
    )
    ax.set_title(f"Method Timing Heatmap (n={sample_size:,})\n(values in milliseconds)")
    ax.set_xlabel("Method Family")
    ax.set_ylabel("Method")
    plt.tight_layout()
    return fig

if baseline and len(df) > 0:
    plot_performance_heatmap(df)
    plt.show()

## 3. Scaling Analysis

In [ ]:
def plot_scaling_by_family(df: pd.DataFrame):
    """Plot scaling behavior for each method family."""
    families = df["family"].unique()
    n_cols = 3
    n_rows = (len(families) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    
    for idx, family in enumerate(sorted(families)):
        ax = axes[idx]
        family_df = df[df["family"] == family]
        
        for method in family_df["method"].unique():
            method_df = family_df[family_df["method"] == method].sort_values("n")
            ax.plot(
                method_df["n"], 
                method_df["median_ms"],
                marker="o",
                label=method
            )
        
        ax.set_xlabel("Sample Size (n)")
        ax.set_ylabel("Time (ms)")
        ax.set_title(f"{family.upper()}")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(families), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle("Scaling Behavior by Method Family", fontsize=14, y=1.02)
    plt.tight_layout()
    return fig

if baseline and len(df) > 0:
    plot_scaling_by_family(df)
    plt.show()

## 4. Speed Category Distribution

In [ ]:
def plot_speed_categories(df: pd.DataFrame):
    """Plot distribution of methods by speed category."""
    # Use largest sample size for classification
    max_n = df["n"].max()
    subset = df[df["n"] == max_n]
    
    category_order = ["fast", "medium", "slow", "very_slow"]
    colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Overall distribution
    category_counts = subset["speed_category"].value_counts().reindex(category_order, fill_value=0)
    axes[0].bar(category_counts.index, category_counts.values, color=colors)
    axes[0].set_xlabel("Speed Category")
    axes[0].set_ylabel("Number of Methods")
    axes[0].set_title(f"Method Distribution by Speed (n={max_n:,})")
    
    # Add count labels
    for i, v in enumerate(category_counts.values):
        axes[0].text(i, v + 0.5, str(v), ha="center", fontweight="bold")
    
    # By family
    family_speed = pd.crosstab(subset["family"], subset["speed_category"])
    family_speed = family_speed.reindex(columns=category_order, fill_value=0)
    family_speed.plot(kind="barh", stacked=True, ax=axes[1], color=colors)
    axes[1].set_xlabel("Number of Methods")
    axes[1].set_ylabel("Method Family")
    axes[1].set_title("Speed Categories by Family")
    axes[1].legend(title="Category", bbox_to_anchor=(1.02, 1))
    
    plt.tight_layout()
    return fig

if baseline and len(df) > 0:
    plot_speed_categories(df)
    plt.show()

## 5. Memory Usage Analysis

In [ ]:
def plot_memory_vs_time(df: pd.DataFrame):
    """Plot memory usage vs computation time."""
    max_n = df["n"].max()
    subset = df[df["n"] == max_n].copy()
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Create scatter with family colors
    families = subset["family"].unique()
    palette = sns.color_palette("husl", len(families))
    
    for idx, family in enumerate(sorted(families)):
        family_df = subset[subset["family"] == family]
        ax.scatter(
            family_df["median_ms"],
            family_df["memory_kb"] / 1024,  # Convert to MB
            label=family,
            alpha=0.7,
            s=100,
            color=palette[idx]
        )
        
        # Annotate outliers (top 10% by time)
        threshold = family_df["median_ms"].quantile(0.9)
        for _, row in family_df[family_df["median_ms"] >= threshold].iterrows():
            ax.annotate(
                row["method"],
                (row["median_ms"], row["memory_kb"] / 1024),
                fontsize=8,
                alpha=0.8
            )
    
    ax.set_xlabel("Computation Time (ms)")
    ax.set_ylabel("Peak Memory (MB)")
    ax.set_title(f"Time vs Memory Trade-off (n={max_n:,})")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.legend(bbox_to_anchor=(1.02, 1), title="Family")
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

if baseline and len(df) > 0:
    plot_memory_vs_time(df)
    plt.show()

## 6. Top 10 Fastest and Slowest Methods

In [ ]:
def show_extremes(df: pd.DataFrame, sample_size: int = 1000):
    """Show fastest and slowest methods."""
    subset = df[df["n"] == sample_size].copy()
    subset = subset.sort_values("median_ms")
    
    print(f"\n{'='*60}")
    print(f"TOP 10 FASTEST METHODS (n={sample_size:,})")
    print(f"{'='*60}")
    fastest = subset.head(10)[["family", "method", "median_ms", "speed_category"]]
    print(fastest.to_string(index=False))
    
    print(f"\n{'='*60}")
    print(f"TOP 10 SLOWEST METHODS (n={sample_size:,})")
    print(f"{'='*60}")
    slowest = subset.tail(10)[["family", "method", "median_ms", "speed_category"]]
    print(slowest.to_string(index=False))

if baseline and len(df) > 0:
    show_extremes(df)

## 7. Summary Statistics

In [ ]:
def summarize_benchmarks(df: pd.DataFrame):
    """Generate summary statistics."""
    max_n = df["n"].max()
    subset = df[df["n"] == max_n]
    
    summary = subset.groupby("family").agg({
        "method": "count",
        "median_ms": ["mean", "min", "max"],
        "memory_kb": "mean",
    }).round(2)
    
    summary.columns = ["n_methods", "avg_ms", "min_ms", "max_ms", "avg_memory_kb"]
    summary = summary.sort_values("avg_ms")
    
    print(f"\nSummary by Family (n={max_n:,})\n")
    print(summary.to_string())
    
    return summary

if baseline and len(df) > 0:
    summary = summarize_benchmarks(df)

## 8. Interactive Method Selection Guide

Based on performance characteristics, here's a quick reference for method selection:

In [ ]:
def method_selection_guide(df: pd.DataFrame):
    """Generate method selection recommendations."""
    max_n = df["n"].max()
    subset = df[df["n"] == max_n]
    
    print("\n" + "="*70)
    print("METHOD SELECTION GUIDE (Performance-Based)")
    print("="*70)
    
    print("\nFAST METHODS (< 10ms at n=1000) - Good for iteration:")
    fast = subset[subset["speed_category"] == "fast"][["family", "method", "median_ms"]]
    if len(fast) > 0:
        for _, row in fast.iterrows():
            print(f"  - {row['family']}/{row['method']}: {row['median_ms']:.1f}ms")
    else:
        print("  (None at this sample size)")
    
    print("\nMEDIUM METHODS (10-100ms) - Production viable:")
    medium = subset[subset["speed_category"] == "medium"][["family", "method", "median_ms"]]
    for _, row in medium.head(10).iterrows():
        print(f"  - {row['family']}/{row['method']}: {row['median_ms']:.1f}ms")
    if len(medium) > 10:
        print(f"  ... and {len(medium) - 10} more")
    
    print("\nSLOW METHODS (> 100ms) - Use sparingly:")
    slow = subset[subset["speed_category"].isin(["slow", "very_slow"])][["family", "method", "median_ms"]]
    for _, row in slow.iterrows():
        print(f"  - {row['family']}/{row['method']}: {row['median_ms']:.1f}ms")

if baseline and len(df) > 0:
    method_selection_guide(df)

---

## Summary

This notebook provides comprehensive performance analysis of all causal inference methods.
Use the visualizations and tables to:

1. **Select appropriate methods** based on computational budget
2. **Understand scaling** behavior for large datasets
3. **Identify bottlenecks** in analysis pipelines
4. **Track regressions** against the golden baseline